# Chốt protocol và ghi thông tin môi trường

## 1. Xác định project và cấu hình
**Vai trò:** đọc đúng file config và bản ghi xác nhận; tính fingerprint để phát hiện thay đổi sau khi chốt. Cell này không import code model/data/training.

In [6]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import hashlib
import importlib.metadata as metadata
import json
import os
import platform
import shutil
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (p for p in candidates
     if (p / "configs/default.json").is_file()
     and (p / "src/cifar_compare/data.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Mở notebook từ project hoặc thư mục notebooks của project.")

REPORTS = PROJECT_ROOT / "reports"
CONFIG_PATH = PROJECT_ROOT / "configs/default.json"
CONFIRMATIONS_PATH = REPORTS / "protocol_confirmations.json"
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8-sig"))
confirmations = json.loads(CONFIRMATIONS_PATH.read_text(encoding="utf-8-sig"))

def sha256_file(path):
    text = Path(path).read_text(encoding="utf-8-sig")
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

config_sha256 = sha256_file(CONFIG_PATH)
data_py_sha256 = sha256_file(PROJECT_ROOT / "src/cifar_compare/data.py")
config_matches_confirmation = config_sha256 == confirmations["config_sha256"]
data_code_matches_confirmation = data_py_sha256 == confirmations["data_py_sha256"]

print("Project:", PROJECT_ROOT)
print("Python executable:", sys.executable)
print("Config:", CONFIG_PATH)
print("Config SHA256:", config_sha256)
print("Khớp cấu hình đã ghi nhận:", config_matches_confirmation)
print("Khớp data.py tại thời điểm chốt:", data_code_matches_confirmation)
print("Xác nhận dữ liệu:", confirmations["data_pipeline"]["status"])
print("Xác nhận metrics/timing:", confirmations["evaluation_and_profiling"]["status"])
print("Training:", {k: config[k] for k in (
    "seed", "input_size", "batch_size", "num_workers", "epochs",
    "learning_rate", "head_learning_rate", "amp", "device"
)})
print("Profiling:", {k: config[k] for k in ("warmup", "repeats")})

Project: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification
Python executable: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification\.venv-1\Scripts\python.exe
Config: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification\configs\default.json
Config SHA256: e40fd090557395ea2dae4dc162abdae3a33c556bd4d290bd96535b3f4bec236e
Khớp cấu hình đã ghi nhận: True
Khớp data.py tại thời điểm chốt: True
Xác nhận dữ liệu: confirmed_via_user
Xác nhận metrics/timing: confirmed_via_user
Training: {'seed': 42, 'input_size': 224, 'batch_size': 32, 'num_workers': 4, 'epochs': 10, 'learning_rate': 0.0001, 'head_learning_rate': 0.001, 'amp': True, 'device': 'cuda'}
Profiling: {'warmup': 20, 'repeats': 100}


## 2. Đối chiếu split đã lưu
**Vai trò:** kiểm tra metadata, số lượng và indices của split mà hai model sẽ dùng chung. Chỉ đọc JSON; không nạp ảnh/test set và không chạy DataLoader. Điều này không thay thế các kiểm tra batch trong notebook 01.

In [7]:
split_path = PROJECT_ROOT / config["split_path"]
split = json.loads(split_path.read_text(encoding="utf-8-sig"))
train_indices = split["train_indices"]
val_indices = split["val_indices"]
train_set = set(train_indices)
val_set = set(val_indices)

split_checks = {
    "seed_42": split["seed"] == config["seed"] == 42,
    "val_size_5000": split["val_size"] == config["val_size"] == 5000,
    "train_count_45000": len(train_indices) == 45000,
    "val_count_5000": len(val_indices) == 5000,
    "unique_train": len(train_set) == len(train_indices),
    "unique_validation": len(val_set) == len(val_indices),
    "no_overlap": not train_set.intersection(val_set),
    "full_train_coverage": train_set.union(val_set) == set(range(50000)),
}
split_ok = all(split_checks.values())
for check, passed in split_checks.items():
    print(("PASS" if passed else "FAIL"), check)
print("Split SHA256:", sha256_file(split_path))
print("Chỉ đối chiếu chỉ số; chưa đọc ảnh hay chạy model.")

PASS seed_42
PASS val_size_5000
PASS train_count_45000
PASS val_count_5000
PASS unique_train
PASS unique_validation
PASS no_overlap
PASS full_train_coverage
Split SHA256: 4d6d124366adedc2d7d229a87d0b19d9bc0f9cae5070d1fa24f46e4cfbaaa5cc
Chỉ đối chiếu chỉ số; chưa đọc ảnh hay chạy model.


## 3. Ghi môi trường từ kernel đang dùng
**Vai trò:** lấy Python, torch/torchvision, CUDA build, GPU, driver và phiên bản các gói thực cài. `nvidia-smi` chỉ truy vấn thông tin GPU; không đo hiệu năng.

In [8]:
import torch
import torchvision

captured_at = datetime.now(timezone(timedelta(hours=8))).isoformat(timespec="seconds")
expected_python = PROJECT_ROOT / ".venv-1/Scripts/python.exe"
try:
    correct_kernel = expected_python.is_file() and os.path.samefile(
        sys.executable, expected_python
    )
except OSError:
    correct_kernel = False

def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "NOT_INSTALLED"

cuda_available = bool(torch.cuda.is_available())
gpus = []
if cuda_available:
    for gpu_index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(gpu_index)
        gpus.append({
            "index": gpu_index,
            "name": torch.cuda.get_device_name(gpu_index),
            "capability": list(torch.cuda.get_device_capability(gpu_index)),
            "total_memory_bytes": int(properties.total_memory),
        })

smi = shutil.which("nvidia-smi")
if smi is None:
    fallback_smi = Path(os.environ.get("SystemRoot", "C:/Windows")) / "System32/nvidia-smi.exe"
    if fallback_smi.is_file():
        smi = str(fallback_smi)

driver_info = {"available": False, "output": "", "error": ""}
if smi is not None:
    try:
        response = subprocess.run(
            [smi, "--query-gpu=index,name,driver_version,memory.total",
             "--format=csv,noheader"],
            capture_output=True, text=True, encoding="utf-8",
            errors="replace", timeout=15,
        )
        driver_info = {
            "available": response.returncode == 0 and bool(response.stdout.strip()),
            "output": response.stdout.strip(),
            "error": response.stderr.strip() if response.returncode else "",
        }
    except (OSError, subprocess.TimeoutExpired) as error:
        driver_info["error"] = str(error)
else:
    driver_info["error"] = "Không tìm thấy nvidia-smi; chưa ghi được driver."

main_packages = {
    name: package_version(name)
    for name in ("torch", "torchvision", "numpy", "scikit-learn",
                 "ipykernel", "matplotlib")
}
active_cuda_device = int(torch.cuda.current_device()) if cuda_available else None
planned_gpu = (
    active_cuda_device is not None
    and "5070ti" in gpus[active_cuda_device]["name"].replace(" ", "").lower()
)
environment_checks = {
    "kernel_is_project_venv1": bool(correct_kernel),
    "torch_cuda_available": cuda_available,
    "pytorch_has_cuda_build": torch.version.cuda is not None,
    "planned_gpu_detected": planned_gpu,
    "driver_recorded": bool(driver_info["available"]),
    "ipykernel_installed": main_packages["ipykernel"] != "NOT_INSTALLED",
}
environment_ok = all(environment_checks.values())
environment = {
    "captured_at": captured_at,
    "timezone": "Asia/Singapore",
    "python_version": platform.python_version(),
    "python_full_version": sys.version.replace("\n", " "),
    "python_executable": sys.executable,
    "expected_kernel_executable": str(expected_python),
    "platform": platform.platform(),
    "torch_version": str(torch.__version__),
    "torchvision_version": str(torchvision.__version__),
    "cuda_build": torch.version.cuda,
    "cuda_available": cuda_available,
    "gpus": gpus,
    "active_cuda_device": active_cuda_device,
    "driver_query": driver_info,
    "packages": main_packages,
    "current_backend_flags": {
        "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
        "cudnn_deterministic": bool(torch.backends.cudnn.deterministic),
        "tf32_matmul": bool(torch.backends.cuda.matmul.allow_tf32),
        "tf32_cudnn": bool(torch.backends.cudnn.allow_tf32),
    },
    "environment_checks": environment_checks,
    "training_environment_ready": environment_ok,
    "visualize_dependency_ready": main_packages["matplotlib"] != "NOT_INSTALLED",
    "note": "Flags ghi nhận từ kernel hiện tại; train.py có thể đặt lại cuDNN khi set_seed. Không có training/benchmark trong notebook này.",
}
REPORTS.mkdir(parents=True, exist_ok=True)
(REPORTS / "environment_phuc.json").write_text(
    json.dumps(environment, ensure_ascii=False, indent=2), encoding="utf-8"
)
lines = [
    "MÔI TRƯỜNG THỰC TẾ CỦA PHÚC",
    "Thời điểm: " + captured_at,
    "Python: " + environment["python_full_version"],
    "Kernel executable: " + sys.executable,
    "Đúng .venv-1: " + str(correct_kernel),
    "OS: " + environment["platform"],
    "torch: " + str(torch.__version__),
    "torchvision: " + str(torchvision.__version__),
    "PyTorch CUDA build: " + str(torch.version.cuda),
    "CUDA available: " + str(cuda_available),
    "GPU: " + json.dumps(gpus, ensure_ascii=False),
    "Driver query columns: index,name,driver_version,memory.total",
    driver_info["output"] or driver_info["error"],
    "Gói chính: " + json.dumps(main_packages, ensure_ascii=False),
    "Điều kiện môi trường: " + json.dumps(environment_checks, ensure_ascii=False),
    "Training environment ready: " + str(environment_ok),
    "Visualize dependency ready: " + str(environment["visualize_dependency_ready"]),
    "Không chạy training, inference hoặc benchmark.",
    "Danh sách gói đầy đủ: reports/requirements_phuc_snapshot.txt",
]
(REPORTS / "environment_phuc.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")

installed = sorted({
    dist.metadata["Name"] + "==" + dist.version
    for dist in metadata.distributions()
    if dist.metadata.get("Name")
}, key=str.lower)
snapshot_header = (
    "# Snapshot gói thực cài từ kernel; không chứa URL/direct credentials.\n"
    "# Các build CUDA cần đúng wheel index khi cài lại; snapshot không thay requirements.txt.\n"
)
(REPORTS / "requirements_phuc_snapshot.txt").write_text(
    snapshot_header + "\n".join(installed) + "\n", encoding="utf-8"
)

print("Python:", environment["python_version"])
print("Kernel đúng .venv-1:", correct_kernel)
print("torch / torchvision:", torch.__version__, "/", torchvision.__version__)
print("CUDA build / available:", torch.version.cuda, "/", cuda_available)
print("Driver:", driver_info["output"] or driver_info["error"])
print("Điều kiện môi trường:", environment_checks)
print("Đã lưu:", REPORTS / "environment_phuc.txt")
print("Đã lưu:", REPORTS / "environment_phuc.json")
print("Đã lưu:", REPORTS / "requirements_phuc_snapshot.txt")
if not environment["visualize_dependency_ready"]:
    print("Ghi chú: matplotlib chưa cài; visualize chưa sẵn sàng. Cell không tự cài thư viện.")

Python: 3.14.4
Kernel đúng .venv-1: True
torch / torchvision: 2.14.1+cu130 / 0.29.1+cu130
CUDA build / available: 13.0 / True
Driver: 0, NVIDIA GeForce RTX 5070 Ti, 617.42, 16303 MiB
Điều kiện môi trường: {'kernel_is_project_venv1': True, 'torch_cuda_available': True, 'pytorch_has_cuda_build': True, 'planned_gpu_detected': True, 'driver_recorded': True, 'ipykernel_installed': True}
Đã lưu: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification\reports\environment_phuc.txt
Đã lưu: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification\reports\environment_phuc.json
Đã lưu: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification\reports\requirements_phuc_snapshot.txt
Ghi chú: matplotlib chưa cài; visualize chưa sẵn sàng. Cell không tự cài thư viện.


## 4. Tổng hợp để gửi lại

In [9]:
readiness_checks = {
    "config_matches_confirmed_fingerprint": config_matches_confirmation,
    "data_code_matches_confirmed_fingerprint": data_code_matches_confirmation,
    "saved_split_valid": split_ok,
    "khang_confirmation_recorded": confirmations["data_pipeline"]["status"] == "confirmed_via_user",
    "quy_confirmation_recorded": confirmations["evaluation_and_profiling"]["status"] == "confirmed_via_user",
    "training_environment_ready": environment_ok,
}
phase2_ready = all(readiness_checks.values())
phase2_report = {
    "captured_at": captured_at,
    "protocol_id": confirmations["protocol_id"],
    "config_sha256": config_sha256,
    "data_py_sha256": data_py_sha256,
    "split_sha256": sha256_file(split_path),
    "checks": readiness_checks,
    "phase2_ready": phase2_ready,
    "visualize_dependency_ready": environment["visualize_dependency_ready"],
    "training_run_started": False,
    "next": "Assistant xem kết quả để xác nhận T012; sau đó tiếp tục T013–T015.",
    "note": "Đây không phải kết quả test, benchmark hoặc kiểm chứng training loop mới.",
}
(REPORTS / "phase2_readiness.json").write_text(
    json.dumps(phase2_report, ensure_ascii=False, indent=2), encoding="utf-8"
)
for check, passed in readiness_checks.items():
    print(("PASS" if passed else "FAIL"), "-", check)
print("\nPHASE 2:", "PASS" if phase2_ready else "CHƯA ĐẠT — gửi lại output/traceback để xem")
print("Visualize dependency ready:", environment["visualize_dependency_ready"])
print("Đã lưu:", REPORTS / "phase2_readiness.json")
print("Hãy Ctrl+S notebook rồi gửi output cell cuối. Chưa chạy training.")

PASS - config_matches_confirmed_fingerprint
PASS - data_code_matches_confirmed_fingerprint
PASS - saved_split_valid
PASS - khang_confirmation_recorded
PASS - quy_confirmation_recorded
PASS - training_environment_ready

PHASE 2: PASS
Visualize dependency ready: False
Đã lưu: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification\reports\phase2_readiness.json
Hãy Ctrl+S notebook rồi gửi output cell cuối. Chưa chạy training.
